# 自回归生成与解码

> 上一部分讲完了训练，模型已经学会给词表里每个 Token 打分。但是训练结束开始生成时，模型每次前向只交出一组 logits，离真正的文字还差最后一步，就是从几万个候选里选出下一个 Token。那么分数怎么变成文字呢？这就是本章要回答的问题。
>
> 本章介绍解码策略的四组核心内容：
>
> 1. **从 logits 到概率**：softmax 把任意分数变成合法的概率分布，这是所有选择规则的地基。
> 2. **三种基本策略**：Greedy 永远选最高分；Temperature 调整分布形状；Top-k / Top-p 裁掉长尾候选。
> 3. **两个常见补丁**：Repetition Penalty 治复读；Beam Search 同时考虑多条路径。
> 4. **完整拼装**：这些规则在一次真实采样里按什么顺序生效。
>
> 读完后，你应该能看懂 Hugging Face、vLLM、SGLang 文档里的 `temperature`、`top_p`、`top_k`、`repetition_penalty` 各自在改什么，说到底它们改变的都是「怎么选」，而不是模型本身。

先把 logits 具体化，我们来看一个具体的例子。假设词表里只有 6 个候选，模型刚读完「法国的首都是」，输出这样一组分数：

```text
巴黎   4.2
伦敦   3.8
北京   1.2
东京   0.8
香蕉  -0.5
。     0.4
```

看到这组分数，你可能会说直接选「巴黎」就可以了。但是事情没这么简单，因为我们有时候希望模型别每次都给相同的回答，有时候模型会陷入复读，有时候反而选择第二名的分数效果更好。这些需求会一个个变成具体的「选择规则」。那么，我们先从最基本的问题开始：logits 到底怎么读？

## 1. 从 Logits 到概率

logits 是没有归一化的分数。看上面那张表就能发现它的两个特点，全部加起来不等于 1，还允许负数，「香蕉」的 -0.5 表示模型认为它非常不合适。所以 logits 只能比大小，不能当概率用。

把分数变成概率靠 softmax，每个 logit 先取指数，再除以所有指数的和。取指数做了两件事，把任何实数变成正数，并且放大分数之间的差距。「巴黎」和「伦敦」的 logit 只差 0.4，softmax 之后「巴黎」的概率恰好是「伦敦」的 $e^{0.4} \approx 1.5$ 倍；而「香蕉」看似只比「。」低 0.9 分，概率上却几乎出局。

In [ ]:
import torch
import torch.nn.functional as F

tokens = ["巴黎", "伦敦", "北京", "东京", "香蕉", "。"]
logits = torch.tensor([4.2, 3.8, 1.2, 0.8, -0.5, 0.4])

probs = F.softmax(logits, dim=-1)
for t, l, p in zip(tokens, logits, probs):
    print(f"{t:>4}  logit={l:>4.1f}  p={p.item():.3f}")

print()
print(f"关键观察：巴黎/伦敦 logit 只差 0.4，概率比值却是 "
      f"{float(probs[0] / probs[1]):.2f} 倍（= e^0.4）")
print(f"香蕉 logit=-0.5 看着不大，概率只剩 {float(probs[4]):.4f}——指数放大了差距")

两个细节值得停一下。第一，「香蕉」的 -0.5 和「。」的 0.4 看起来只差不到 1 分，概率上却是 9 倍的差距，换句话说 logit 的「分差」和概率的「倍差」不是一回事。第二，概率加起来正好是 1，从这一步开始，「选 Token」就变成了「按概率抽奖」。怎么抽，就是接下来所有策略要回答的问题。

## 2. Greedy 解码

最直接的选择规则就是每次都拿概率最高的 Token，用 `argmax` 一步到位。它有两个实打实的优点，结果稳定意味着同样的输入永远得到同样的输出方便复现和调试，计算最省意味着没有任何额外步骤。闭卷问答、代码补全这类要的就是那个正确答案的场景，greedy 往往就是默认配置。但稳定也意味着单调，如果让模型写一个关于春天的开头，greedy 每次都给出同一句哪怕这句平平无奇。概率第二名的候选可能同样合理，却永远没有被选上的机会。带着这个遗憾，我们先看第一种改进：在抽签之前，先调整概率本身的形状。

In [ ]:
next_id = torch.argmax(logits).item()
print("Greedy 选择:", tokens[next_id])

print()
print("关键观察：argmax 是确定性的——同样的 logits，重复一万次也永远选同一个 Token")
print("第二名「伦敦」哪怕只差 0.4 分，也永远没有出场机会")

## 3. Temperature 与分布形状

Temperature 只做一件事，就是在 softmax 之前把所有 logits 除以一个数 $T$，公式如下。

$$
p_i = \mathrm{softmax}(z_i / T)
$$

除以一个小于 1 的数等于放大分数差距，分布变尖模型更笃定行为接近 greedy；除以一个大于 1 的数等于压平差距，分布变平低分候选也分到概率输出更多样。有一句话值得先说在前面，Temperature 不会删除任何候选，无论温度多高「香蕉」的概率都不会变成零只是被重新分配了大小。这一点马上会引出新的问题。

In [ ]:
for T in [0.2, 0.7, 1.0, 1.5]:
    p = F.softmax(logits / T, dim=-1)
    print(f"T={T:<3}: " + ", ".join(f"{t}:{x:.2f}" for t, x in zip(tokens, p.tolist())))

print()
print("关键观察：T=0.2 时前两名接近垄断；T=1.5 时候选差距被压平")
print("但无论 T 多大，香蕉的概率都不为 0——高温把长尾放出来了")

## 4. Top-k 与 Top-p 截断

Temperature 调高之后一个副作用出现了，原本概率接近零的候选也分到了不小的概率质量。一次抽签抽中香蕉开头的句子整段生成就毁了。想让合理的候选机会更均等同时把明显不合理的候选挡在门外，这就是截断要做的事。

两种裁法。Top-k 最简单，只保留分数最高的 k 个候选其余全部出局（实现上把 logit 设成 $-\infty$ softmax 之后概率为零），k=2 就是只在巴黎伦敦两个里抽。

Top-p 也叫 Nucleus Sampling 解决 Top-k 的一个死板之处，因为不同 Prompt 的分布形状差别很大。模型很确定时前两名可能就占了 95% 的概率 k=5 反而把不靠谱的候选放进来了；模型很犹豫时前 20 名概率都差不多 k=5 又裁得太狠。Top-p 不数候选的个数改数概率的质量，把候选按概率从高到低累加保留累计概率刚好盖住 p 的最小集合。

用一组数字走一遍这个累加过程：

```text
A 0.50   累计 0.50
B 0.25   累计 0.75
C 0.15   累计 0.90  ← 盖住 0.90，到此为止
D 0.06   累计 0.96
...
```

`top_p = 0.9` 时保留 A、B、C 三个 D 及之后全部出局。分布尖的时候它自动少留几个分布平的时候自动多留几个，这就是动态的含义。

In [ ]:
def top_k_filter(logits, k):
    if k is None or k >= logits.numel():
        return logits
    threshold = torch.topk(logits, k).values[-1]
    return torch.where(logits < threshold, torch.tensor(float("-inf")), logits)

def top_p_filter(logits, p):
    sorted_logits, sorted_idx = torch.sort(logits, descending=True)
    sorted_probs = F.softmax(sorted_logits, dim=-1)
    cumulative = torch.cumsum(sorted_probs, dim=-1)
    remove = cumulative > p
    remove[1:] = remove[:-1].clone()
    remove[0] = False
    sorted_logits[remove] = float("-inf")
    out = torch.full_like(logits, float("-inf"))
    out[sorted_idx] = sorted_logits
    return out

cases = [
    ("top_k=2", top_k_filter(logits.clone(), 2)),
    ("top_p=0.9", top_p_filter(logits.clone(), 0.9)),
]
for name, filtered in cases:
    p = F.softmax(filtered, dim=-1)
    kept = [(t, round(x, 3)) for t, x in zip(tokens, p.tolist()) if x > 0]
    print(name, kept)

print()
print("关键观察：top_k=2 只剩巴黎伦敦；top_p=0.9 留到累计概率盖住 0.9 为止")

## 5. Repetition Penalty

前面所有策略有一个共同的盲区，它们只看当前这一步的概率不知道前面已经生成过什么。于是会出现一种经典失败，比如模型反复写同一个词。一旦「非常好」成为高分候选 greedy 会永远选它 sampling 也可能连续抽中它。问题不在模型在选择规则，没有任何机制惩罚刚说过的词。

具体来说，它用简单的数学操作实现。Repetition Penalty 补上这个机制，对已经出现过的 Token 把它的 logit 压低一点。Hugging Face 的实现规则是正 logit 除以惩罚系数 $c$ 负 logit 乘以 $c$（$c>1$ 只作用于出现过的 Token），公式如下。

$$
z_i' = \begin{cases} z_i / c & z_i > 0 \\ z_i \cdot c & z_i \le 0 \end{cases}
$$

先手算一个关键情形，「巴黎」已经出现过一次取 $c=1.3$。它的 logit 4.2 被压成 4.2/1.3≈3.23 而没出现过的「伦敦」保持 3.8，压完之后第一名和第二名交换下一次选择自然翻转。penalty 同样不删除候选只是让说过的暂时吃亏。

In [ ]:
def apply_repetition_penalty(logits, generated_ids, penalty):
    """压低已出现 token 的 logit：正数除以 penalty，负数乘以 penalty"""
    z = logits.clone()
    for i in set(generated_ids):
        if z[i] > 0:
            z[i] = z[i] / penalty
        else:
            z[i] = z[i] * penalty
    return z

generated_ids = [tokens.index("巴黎")]  # 「巴黎」已经出现过一次
z_new = apply_repetition_penalty(logits, generated_ids, penalty=1.3)

for i, (t, old, new) in enumerate(zip(tokens, logits, z_new)):
    mark = "  <- 已出现，被压低" if i in generated_ids else ""
    print(f"{t:>3}  原始 {old:>5.2f}  压后 {new:>5.2f}{mark}")

print()
print("关键观察：argmax 从「", tokens[logits.argmax()], "」变成「", tokens[z_new.argmax()], "」")

In [ ]:
# 图里看更直观：只有出现过的 Token 被压低
import matplotlib.pyplot as plt

labels = ["Paris", "London", "Beijing", "Tokyo", "banana", "."]
xs = range(len(labels))
width = 0.38

plt.figure(figsize=(7, 3.5))
plt.bar([i - width / 2 for i in xs], logits.tolist(), width=width,
        label="original logits")
plt.bar([i + width / 2 for i in xs], z_new.tolist(), width=width,
        label="after repetition penalty")
plt.xticks(list(xs), labels)
plt.ylabel("logit")
plt.title("Only the repeated token (Paris) gets pushed down")
plt.legend()
plt.show()

## 6. Beam Search

到这里为止的所有策略都有一个共同假设，每一步选局部最优整体输出就不会差。真的如此吗？我们看一个只有两步的例子。

```text
Step 1:  A: 0.6    B: 0.4
Step 2:  A 之后: x: 0.5 / y: 0.5
         B 之后: x: 0.9 / y: 0.1
```

Greedy 第一步选 A（0.6 更高），之后无论怎么选整体概率最多 0.6 × 0.5 = 0.30。但是第一步的第二名 B 分数是 0.4，如果后面接上高分 x 算出来 0.4 × 0.9 = 0.36，有时候反而选择第二名的分数效果更好。换句话说局部最优不等于全局最优，这就是 Beam Search 存在的理由。

做法上每一步不再只保留 1 个候选，而是同时保留 k 条备选序列（k 叫 beam width），用整体累计得分互相比较最后输出总分最高的一条。工程实现里累计得分用 log 概率相加而不是概率连乘，因为一连串小数相乘会下溢取对数变成加法就稳定了，这里数字小直接乘就能看清楚。

那为什么现在的聊天模型几乎不用它？Beam Search 假设整体概率最高的输出就是最好的输出，这在翻译摘要这类有标准答案的任务上通常成立；但开放式聊天里概率最高的回答往往是最平最安全的那个多样性反而没了。所以现代对话生成默认走 sampling，beam search 主要留在封闭任务里。

In [ ]:
# 用同一组数字验证：第一步的第一名，不一定是整条路的赢家
step1 = {"A": 0.6, "B": 0.4}
step2 = {"A": {"x": 0.5, "y": 0.5}, "B": {"x": 0.9, "y": 0.1}}

greedy_first = max(step1, key=step1.get)
greedy_total = step1[greedy_first] * max(step2[greedy_first].values())
print(f"Greedy 第一步选 {greedy_first}，之后接着走局部最优，总分 {greedy_total:.2f}")

all_paths = {(t1, t2): p1 * p2
             for t1, p1 in step1.items()
             for t2, p2 in step2[t1].items()}
best_path = max(all_paths, key=all_paths.get)
print("所有完整路径:", all_paths)
print("关键观察：整条路的最优是", "".join(best_path),
      "总分", all_paths[best_path])

In [ ]:
# 把这棵两步搜索树画出来：线越粗概率越高，绿色是 Beam 找到的整条最优路径
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7, 4))
nodes = {"start": (0, 0.5), "A": (1, 0.78), "B": (1, 0.22),
         "Ax": (2, 0.95), "Ay": (2, 0.62), "Bx": (2, 0.38), "By": (2, 0.05)}
edges = [("start", "A", 0.6), ("start", "B", 0.4),
         ("A", "Ax", 0.5), ("A", "Ay", 0.5),
         ("B", "Bx", 0.9), ("B", "By", 0.1)]

for a, b, p in edges:
    (x1, y1), (x2, y2) = nodes[a], nodes[b]
    on_best = (a, b) in {("start", "B"), ("B", "Bx")}
    ax.plot([x1, x2], [y1, y2], color="tab:green" if on_best else "tab:gray",
            linewidth=1 + 8 * p, alpha=1.0 if on_best else 0.45, zorder=1)
    ax.text((x1 + x2) / 2, (y1 + y2) / 2 + 0.04, f"{p:.1f}",
            ha="center", fontsize=9)

for name, (x, y) in nodes.items():
    ax.text(x, y, name, ha="center", va="center", zorder=2,
            bbox=dict(boxstyle="circle,pad=0.25", fc="white", ec="black"))

ax.text(*nodes["Ax"], "  A-x total 0.30\n  (greedy's choice)", va="center", fontsize=9)
ax.text(*nodes["Bx"], "  B-x total 0.36\n  (best, beam finds it)", va="center", fontsize=9)
ax.set_xlim(-0.2, 3.6)
ax.set_ylim(-0.1, 1.15)
ax.axis("off")
ax.set_title("Beam width 2: the best full path may start with the runner-up")
plt.show()

## 7. 一次完整的采样流程

把前面的规则按真实框架的处理顺序串起来，一次 Decode step 大致长这样：

```text
model forward
    ↓
logits
    ↓ repetition / presence / frequency penalties（先处理「历史」）
temperature（再调整形状）
    ↓
top-k / top-p / min-p（最后裁剪）
    ↓
sampling（multinomial 抽签）
    ↓
next token
```

顺序是有意义的，penalty 依赖已经生成过什么必须最先改分数。temperature 和截断都在改分布先后影响不大但都发生在抽签之前。不同框架的 processor 顺序可能有差异生产环境以具体实现为准。

In [ ]:
def sample_next(logits, temperature=1.0, top_k=None, top_p=None, seed=0):
    torch.manual_seed(seed)
    x = logits.clone() / max(temperature, 1e-5)
    x = top_k_filter(x, top_k)
    if top_p is not None:
        x = top_p_filter(x, top_p)
    probs = F.softmax(x, dim=-1)
    return torch.multinomial(probs, 1).item(), probs

print("greedy:", tokens[torch.argmax(logits).item()])
for name, cfg in [
    ("T=0.7, p=0.9", dict(temperature=0.7, top_p=0.9)),
    ("T=1.2, p=0.95", dict(temperature=1.2, top_p=0.95)),
]:
    picks = [tokens[sample_next(logits, seed=s, **cfg)[0]] for s in range(8)]
    print(name, picks)


## 8. 常用生成参数

把这一章出现过的参数放进一张表。以后在 API 文档模型卡招聘 JD 里看到它们先问一句：它作用在哪一层，改分数改形状裁候选还是管停止条件？换句话说搞清楚层级才能明白调参在改什么。

| 参数 | 作用 |
|:---|:---|
| `temperature` | 调整概率分布的形状（尖或平） |
| `top_k` | 只保留分数最高的 k 个候选 |
| `top_p` | 按累计概率质量动态截断 |
| `repetition_penalty` | 压低已出现 Token 的分数 |
| `max_tokens` / `max_new_tokens` | 生成长度上限 |
| `stop` / EOS | 停止条件 |
| `seed` | 采样随机性的种子 |

## 小结

这一章从一组 logits 出发，把怎么选 Token 拆成了一层层规则。首先 softmax 把分数变成概率是一切规则的起点。接下来 Greedy 稳定但单调，Temperature 不删候选只重新分配概率。然后 Top-k 数候选个数 Top-p 数概率质量都在解决高温后长尾混进来。

另外 Repetition Penalty 把出现过的 Token 压低治复读。还有 Beam Search 同时保留 k 条备选序列找全局最优，而开放式聊天更常用 sampling。说到底这些全是选 Token 的规则，从头到尾模型本身一个参数都没动。

参数没动，下一章的问题也随之而来：

> 为了得到这组 logits，模型一次前向到底做了什么？为什么这么慢？

## 作业

三道题分别对应三类选择规则：penalty、截断、Temperature。接下来我们依次看看每道题的要求。

> **关于 AI 辅助**：可以让 AI 提示思路、拆解步骤，但不建议直接让 AI 完成题目。
> 这些规则都不长，自己写一遍印象最深。

### 作业 1：实现 repetition penalty

只压低出现过的 Token，正 logit 除以 penalty 负 logit 乘以 penalty。这个操作很容易实现。小提示是遍历 `generated_ids` 里的下标，按 `z[i]` 的正负号决定做除法还是乘法。

In [ ]:
# 作业 1：repetition penalty 填空

test_logits = torch.tensor([4.2, 3.8, -0.5, 0.4])
test_generated = [0]  # 下标 0 的 token 已经出现过

def penalize(logits, generated_ids, penalty):
    """返回压低已出现 token 之后的新 logits"""
    z = logits.clone()
    for i in set(generated_ids):
        # TODO：把下面三引号里的内容替换成你的代码
        """在这里按正负号对 z[i] 做除法或乘法"""
    return z

result = penalize(test_logits, test_generated, 1.2)
assert torch.isclose(result[0], torch.tensor(4.2 / 1.2)), result
assert torch.isclose(result[1], torch.tensor(3.8)), result
print("✅ 作业 1 通过：你实现了复读抑制的基本规则")

### 作业 2：算出 top-p 保留几个候选

概率已按从高到低排序，top-p 要保留累计概率刚好盖住 p 的最小候选集合。我们算一下就知道了。小提示是从高到低累加，找到第一个让累计和大于等于 p 的位置，保留到这个位置为止。

In [ ]:
# 作业 2：top-p 候选数 填空

probs = torch.tensor([0.50, 0.25, 0.15, 0.06, 0.04])  # 已按从高到低排序

def top_p_keep_count(probs, p):
    """返回 top-p 截断后保留的候选个数"""
    # TODO：把下面三引号里的内容替换成你的代码
    """在这里用累计概率算出需要保留几个候选"""

assert top_p_keep_count(probs, 0.5) == 1
assert top_p_keep_count(probs, 0.9) == 3
assert top_p_keep_count(probs, 0.99) == 5
print("✅ 作业 2 通过：你理解了 top-p 是按概率质量动态截断")

### 作业 3：用熵量化 Temperature 的效果

分布越平采样结果越难预测，熵（entropy）就是量化这件事的标准指标。这是作业里最有趣的一题。小提示是先算 `F.softmax(logits / T, dim=-1)`，熵的定义是 $-\sum_i p_i \log p_i$。

In [ ]:
# 作业 3：不同 Temperature 下的熵 填空

def entropy(probs):
    """计算分布的熵：越大表示分布越平，采样越难预测"""
    p = probs[probs > 0]
    return float(-(p * p.log()).sum())

def probs_at_temperature(T):
    # TODO：把下面三引号里的内容替换成你的代码
    """返回 logits 除以 T 之后 softmax 的概率分布"""

assert entropy(probs_at_temperature(0.3)) < entropy(probs_at_temperature(1.0))
assert entropy(probs_at_temperature(1.0)) < entropy(probs_at_temperature(3.0))
print("✅ 作业 3 通过：低温分布更尖、高温分布更平，你能量化它了")